# Bài 5.8 · GAN và diffusion: hai cách học để sinh mẫu

**Câu hỏi của buổi này:** hai mạng thi nhau (GAN) và một mạng học đoán nhiễu (diffusion) sinh ra mẫu khác nhau thế nào, và đo một model sinh bằng gì khi vài ảnh đẹp chưa đủ?

Dữ liệu: **tám cụm Gaussian trên vòng tròn** (2 chiều, nhìn được từng mẫu), **MNIST** (chữ số viết tay), **CIFAR-10** (ảnh màu 32 × 32). Model: GAN nhỏ và noise predictor nhỏ tự train, cộng **DDPM đã học sẵn** `google/ddpm-cifar10-32`.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu 2D và thước đo | tám cụm, cách đếm mode phủ, khoảng cách sliced Wasserstein |
| 2. GAN trên dữ liệu 2D | kiểm đường gradient, mẫu sinh theo bước train, loss, mode phủ |
| 3. GAN trên MNIST | ảnh sinh theo epoch, khoảng cách Fréchet nhỏ, phân bố chữ số |
| 4. Diffusion thuận | ảnh CIFAR-10 nhòe dần, đồ thị $\bar\alpha_t$, kiểm công thức đóng |
| 5. Noise predictor 2D | train đoán nhiễu, mẫu dịch dần về tám cụm |
| 6. Sinh có điều kiện và guidance | nhãn cụm, hệ số $w$ đổi độ bám nhãn và độ đa dạng |
| 7. DDPM có sẵn khử nhiễu | $x_t$ và ước lượng $x_0$ theo thời gian trên CIFAR-10 |
| 8. Số bước lấy mẫu | DDPM đầy đủ và DDIM 50, 20, 10 bước: thời gian và ảnh |
| 9. Đánh giá model sinh | so GAN và diffusion, ảnh chọn lọc, FD phụ thuộc số mẫu |
| 10. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 6 phút. Bài giảng: [ai-canuc80k · bài 5.8](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/5.8).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, math, subprocess, sys
import torch, torchvision
from torch import nn
import torch.nn.functional as F
from scipy.linalg import sqrtm

try:
    import diffusers
except ImportError:
    with quiet(): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers'], check=True, capture_output=True)
    import diffusers

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
STEPS_2D, EPOCHS_IMG, T_DDPM, N_IMG = 8000, 10, 1000, 8      # cỡ chạy: bước train 2D, epoch MNIST, bước DDPM, số ảnh CIFAR sinh
table(pd.DataFrame([('torch', torch.__version__), ('diffusers', diffusers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu 2D và thước đo

Tám cụm Gaussian (độ lệch chuẩn 0.1) đặt đều trên vòng tròn bán kính 2. Dữ liệu 2 chiều có lợi thế: nhìn thẳng được model sinh đủ cụm hay bỏ sót, thứ mà ảnh khó cho thấy.

Hai thước đo dùng cho mọi mục 2D:

- **Mode phủ**: một mẫu "tốt" nếu cách tâm gần nhất không quá $3\sigma = 0.3$. Một cụm được tính là phủ khi có ít nhất 2.5% số mẫu tốt rơi vào (kỳ vọng mỗi cụm là 12.5%).
- **Sliced Wasserstein (SW)**: chiếu hai tập mẫu lên 200 hướng ngẫu nhiên, sắp xếp từng hướng rồi lấy trung bình độ lệch. Bằng 0 khi hai tập giống hệt; hai tập thật khác nhau vẫn có SW nhỏ dương vì mẫu hữu hạn.

In [ ]:
K, R, SD = 8, 2.0, 0.1
ang = np.arange(K) * 2 * np.pi / K
CEN = torch.tensor(np.c_[R * np.cos(ang), R * np.sin(ang)], dtype=torch.float32)
THR = 3 * SD

def sample_real(n):
    k = torch.randint(0, K, (n,))
    return CEN[k] + SD * torch.randn(n, 2), k

def mode_stats(x):
    d = torch.cdist(x.detach().cpu(), CEN); near, idx = d.min(1); good = near < THR
    share = torch.bincount(idx[good], minlength=K).float() / len(x)
    return {'modes': int((share >= .025).sum()), 'good': good.float().mean().item(), 'share': share}

PROJ = F.normalize(torch.randn(2, 200, generator=torch.Generator().manual_seed(1)), dim=0)
def sw(x, y):
    return ((x.detach().cpu() @ PROJ).sort(0).values - (y.detach().cpu() @ PROJ).sort(0).values).abs().mean().item()

def ring(ax, title, x=None, c=None, s=3):
    for p in CEN: ax.add_patch(plt.Circle(p.numpy(), THR, fill=False, ec=C['gray'], lw=.7, alpha=.7))
    if x is not None: ax.scatter(*x.detach().cpu().T, s=s, alpha=.5, color=c or C['clay'], lw=0)
    ax.set(xlim=(-3, 3), ylim=(-3, 3), xticks=[-2, 0, 2], yticks=[-2, 0, 2], xlabel='x₁', ylabel='x₂'); ax.set_aspect('equal'); ax.set_title(title, fontsize=10)

N = 2000
REF, _ = sample_real(N); REF2, _ = sample_real(N)
base_rows = [('thật, một lần rút khác', REF2), ('chỉ một cụm', CEN[0] + SD * torch.randn(N, 2)), ('nhiễu chuẩn N(0, 1.5²)', 1.5 * torch.randn(N, 2))]
rows = []
for name, x in base_rows:
    m_ = mode_stats(x); rows.append((name, m_['modes'], m_['good'], sw(REF, x)))
SW0 = rows[0][3]
table(pd.DataFrame(rows, columns=['tập mẫu', 'mode phủ (/8)', 'mẫu tốt', 'SW so với tập thật']), 'Thước đo trên ba tập mẫu tham chiếu',
      fmt={'mẫu tốt': '{:.0%}', 'SW so với tập thật': '{:.3f}'})
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, (name, x) in zip(axes, [('thật', REF)] + base_rows[1:]): ring(ax, name, x, C['blue'])
plt.tight_layout(); plt.show()
check('tập thật phủ đủ 8 cụm, SW giữa hai tập thật nhỏ hơn mọi tập tham chiếu sai', rows[0][1] == 8 and SW0 < min(rows[1][3], rows[2][3]))
note(f'Hai tập thật khác nhau vẫn cách nhau SW = <b>{SW0:.3f}</b>: đó là mức sàn do chỉ có {N} mẫu, không model nào xuống dưới được. '
     f'Tập chỉ có một cụm cách {rows[1][3]:.3f} và tập nhiễu chuẩn cách {rows[2][3]:.3f}. Nhưng nhiễu chuẩn có {rows[2][2]:.0%} mẫu tốt mà phủ {rows[2][1]} cụm: một thước đo không đủ, cần đọc cả ba cột.')

## 2. GAN trên dữ liệu 2D

Generator nhận $z$ 8 chiều, discriminator nhận điểm 2 chiều; mỗi mạng là MLP ba lớp ẩn 128. Mỗi vòng lặp có hai cập nhật luân phiên: bước D (thật nhãn 1, giả nhãn 0, `detach` ở ảnh giả) rồi bước G (cho D chấm ảnh giả, đòi nhãn 1, chính là non-saturating loss của bài).

Trước khi train thật, kiểm đường gradient trên đúng một bước: bước D không được để lại gradient hay đổi trọng số trong G, bước G không được đổi trọng số D.

In [ ]:
ZD, H = 8, 128
def mlp(i, o): return nn.Sequential(nn.Linear(i, H), nn.LeakyReLU(.2), nn.Linear(H, H), nn.LeakyReLU(.2), nn.Linear(H, H), nn.LeakyReLU(.2), nn.Linear(H, o))
bce = nn.BCEWithLogitsLoss()
def make_gan():
    G, D = mlp(ZD, 2), mlp(2, 1)
    return G, D, torch.optim.Adam(G.parameters(), 2e-4, betas=(.5, .999)), torch.optim.Adam(D.parameters(), 2e-4, betas=(.5, .999))

def gan_step(G, D, oG, oD, bs=256):
    real, _ = sample_real(bs); fake = G(torch.randn(bs, ZD))
    lD = bce(D(real), torch.ones(bs, 1)) + bce(D(fake.detach()), torch.zeros(bs, 1))
    oD.zero_grad(); lD.backward(); oD.step()
    lG = bce(D(fake), torch.ones(bs, 1))
    oG.zero_grad(); lG.backward(); oG.step()
    return lD.item(), lG.item()

torch.manual_seed(1)
G, D, oG, oD = make_gan()
real, _ = sample_real(256); fake = G(torch.randn(256, ZD))
g_w = G[0].weight.clone(); d_w = D[0].weight.clone()
lD = bce(D(real), torch.ones(256, 1)) + bce(D(fake.detach()), torch.zeros(256, 1))
oD.zero_grad(); lD.backward(); oD.step()
check('bước D: G không có gradient và trọng số G giữ nguyên (nhờ detach)', all(p.grad is None for p in G.parameters()) and torch.equal(g_w, G[0].weight))
check('bước D: trọng số D đã đổi', not torch.equal(d_w, D[0].weight))
d_w = D[0].weight.clone()
lG = bce(D(fake), torch.ones(256, 1)); oG.zero_grad(); lG.backward(); oG.step()
check('bước G: trọng số G đã đổi, trọng số D giữ nguyên', not torch.equal(g_w, G[0].weight) and torch.equal(d_w, D[0].weight))
check(f'D mới khởi tạo chưa phân biệt được: loss D = {lD.item():.3f}, gần 2·ln 2 = {2 * math.log(2):.3f}', abs(lD.item() - 2 * math.log(2)) < .3)

In [ ]:
SNAP = sorted({int(STEPS_2D * f) for f in (0, .02, .08, .25, .5, 1)})
fixed_z = torch.randn(N, ZD)
torch.manual_seed(0)
G, D, oG, oD = make_gan()
snaps, losses, cov = {}, [], []
t0 = time.time()
for s in range(STEPS_2D + 1):
    with torch.no_grad():
        if s in SNAP: snaps[s] = G(fixed_z)
        if s % 50 == 0: cov.append((s, mode_stats(G(fixed_z[:1000]))['modes']))
    if s < STEPS_2D: losses.append(gan_step(G, D, oG, oD))
t_gan2d = time.time() - t0
losses = np.array(losses); cov = np.array(cov)
fin = snaps[STEPS_2D]; ms_g = mode_stats(fin); sw_g = sw(REF, fin)

fig, axes = plt.subplots(1, len(SNAP), figsize=(2.6 * len(SNAP), 3.1))
for ax, s in zip(axes, SNAP):
    ms = mode_stats(snaps[s]); ring(ax, f'bước {s}: {ms["modes"]}/8 cụm', snaps[s], C['clay'])
    if ax is not axes[0]: ax.set(ylabel='', yticklabels=[])
plt.tight_layout(); plt.show()

sm = lambda v, w=100: np.convolve(v, np.ones(w) / w, mode='valid')
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(sm(losses[:, 0]), color=C['blue'], label='loss D'); axes[0].plot(sm(losses[:, 1]), color=C['clay'], label='loss G')
axes[0].axhline(2 * math.log(2), color=C['gray'], ls=':', lw=1); axes[0].text(len(losses) * .98, 2 * math.log(2) + .05, '2·ln 2', ha='right', color=C['gray'], fontsize=9)
axes[0].set(xlabel='bước train', ylabel='loss (trung bình 100 bước)', title='Loss của D và G'); axes[0].legend(loc='upper right')
axes[1].plot(cov[:, 0], cov[:, 1], color=C['green']); axes[1].set(xlabel='bước train', ylabel='số cụm được phủ (/8)', title='Mode phủ theo bước train', ylim=(0, 8.5))
plt.tight_layout(); plt.show()
tail = cov[cov[:, 0] >= .8 * STEPS_2D, 1]
check('sau train GAN phủ nhiều cụm hơn lúc đầu', ms_g['modes'] > cov[0, 1])
dead = [i for i in range(K) if ms_g['share'][i] < .025]
note(f'Sau {STEPS_2D} bước ({t_gan2d:.0f} giây): phủ <b>{ms_g["modes"]}/8</b> cụm, {ms_g["good"]:.0%} mẫu nằm trong 3σ của một tâm, SW = {sw_g:.3f} (sàn {SW0:.3f}). '
     + (f'Cụm bị bỏ: {", ".join(str(i + 1) for i in dead)}; đây là kiểu mode collapse một phần, và các cụm không còn đều nhau (cụm nhiều nhất chiếm {ms_g["share"].max() / max(ms_g["good"], 1e-9):.0%} số mẫu tốt). ' if dead else 'Tám cụm đều có mẫu. ')
     + f'Trong 20% bước cuối số cụm phủ dao động từ {int(tail.min())} đến {int(tail.max())}, nên số ở bước cuối chỉ là một lát cắt của quá trình đang dao động. '
     + 'Loss G không giảm dần như loss supervised: nó đo G đang lừa D tới đâu, mà D cũng đổi theo.')

## 3. GAN trên MNIST

Cùng công thức, mạng lớn hơn: G là MLP 64 → 256 → 512 → 784 với `tanh`, D là MLP 784 → 512 → 256 → 1. Dùng cả 60 000 ảnh train, mỗi epoch xem hết một lượt.

Để đo ảnh sinh bằng một con số, ta train nhanh một **CNN nhận dạng chữ số** (2 epoch) và lấy vector 64 chiều ở lớp áp chót làm feature. **Khoảng cách Fréchet (FD)** giữa feature của ảnh sinh và ảnh thật so hai Gaussian khớp với hai tập feature, cùng ý với FID nhưng dùng feature của CNN nhỏ này thay cho Inception, nên **số này không so được với FID trong bài báo**.

In [ ]:
from torchvision.datasets import MNIST
with quiet():
    tr_ds, te_ds = MNIST('/tmp/mnist', train=True, download=True), MNIST('/tmp/mnist', train=False, download=True)
Xtr = (tr_ds.data[:, None].float() / 255).to(device); Ytr = tr_ds.targets.to(device)
Xte = (te_ds.data[:, None].float() / 255).to(device); Yte = te_ds.targets.to(device)

class Clf(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(1568, 64), nn.ReLU())
        s.out = nn.Linear(64, 10)
    def forward(s, x): return s.out(s.conv(x))
torch.manual_seed(0)
clf = Clf().to(device); opt = torch.optim.Adam(clf.parameters(), 2e-3)
for ep in range(2):
    for idx in torch.randperm(len(Xtr), device=device).split(256):
        loss = F.cross_entropy(clf(Xtr[idx]), Ytr[idx]); opt.zero_grad(); loss.backward(); opt.step()
clf.eval()
@torch.no_grad()
def feats(x): return torch.cat([clf.conv(x[i:i + 2000]) for i in range(0, len(x), 2000)]).cpu().double().numpy()
@torch.no_grad()
def labels(x): return torch.cat([clf(x[i:i + 2000]).argmax(1) for i in range(0, len(x), 2000)]).cpu()
def fd(a, b):
    ma, mb, ca, cb = a.mean(0), b.mean(0), np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    return float(((ma - mb) ** 2).sum() + np.trace(ca) + np.trace(cb) - 2 * np.trace(sqrtm(ca @ cb).real))

acc = (labels(Xte) == Yte.cpu()).float().mean().item()
F_te, F_tr = feats(Xte), feats(Xtr[:10000])
fd_real = fd(F_tr, F_te)
check(f'CNN nhận dạng đạt accuracy {acc:.1%} trên 10 000 ảnh test (cần trên 97%)', acc > .97)

def gen_net(): return nn.Sequential(nn.Linear(64, 256), nn.LeakyReLU(.2), nn.Linear(256, 512), nn.LeakyReLU(.2), nn.Linear(512, 784), nn.Tanh()).to(device)
def dis_net(): return nn.Sequential(nn.Linear(784, 512), nn.LeakyReLU(.2), nn.Linear(512, 256), nn.LeakyReLU(.2), nn.Linear(256, 1)).to(device)
torch.manual_seed(0)
Gm, Dm = gen_net(), dis_net()
oGm = torch.optim.Adam(Gm.parameters(), 2e-4, betas=(.5, .999)); oDm = torch.optim.Adam(Dm.parameters(), 2e-4, betas=(.5, .999))
zf = torch.randn(10000, 64, device=device); zshow = zf[:16]
@torch.no_grad()
def gen_imgs(): return (Gm(zf) + 1).div(2).view(-1, 1, 28, 28)

SNAP_E = sorted({0, 1, max(1, EPOCHS_IMG // 3), EPOCHS_IMG})
grids, fds = {}, [fd(feats(gen_imgs()), F_te)]
grids[0] = gen_imgs()[:16].cpu()
t0 = time.time()
for ep in range(1, EPOCHS_IMG + 1):
    for idx in torch.randperm(len(Xtr), device=device).split(128):
        real = Xtr[idx].view(-1, 784) * 2 - 1; b = len(real); fake = Gm(torch.randn(b, 64, device=device))
        lD = bce(Dm(real), torch.ones(b, 1, device=device)) + bce(Dm(fake.detach()), torch.zeros(b, 1, device=device))
        oDm.zero_grad(); lD.backward(); oDm.step()
        lG = bce(Dm(fake), torch.ones(b, 1, device=device)); oGm.zero_grad(); lG.backward(); oGm.step()
    imgs_ = gen_imgs(); fds.append(fd(feats(imgs_), F_te)); grids[ep] = imgs_[:16].cpu()
t_mnist = time.time() - t0
fds = np.array(fds); cls_share = np.bincount(labels(imgs_).numpy(), minlength=10) / len(imgs_)
t0 = time.time(); gen_imgs(); torch.cuda.synchronize() if device == 'cuda' else None; ms_img_gan = (time.time() - t0) / len(zf) * 1000

fig, axes = plt.subplots(len(SNAP_E), 1, figsize=(11, 1.0 * len(SNAP_E) + .6))
for ax, e in zip(axes, SNAP_E):
    ax.imshow(torchvision.utils.make_grid(grids[e], nrow=16, padding=1, pad_value=1).permute(1, 2, 0).numpy(), cmap='gray'); ax.axis('off')
    ax.text(-.01, .5, 'chưa train' if e == 0 else f'epoch {e}', transform=ax.transAxes, ha='right', va='center', fontsize=10)
plt.suptitle('16 ảnh sinh từ cùng 16 vector z qua các epoch', y=1.0); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(range(EPOCHS_IMG + 1), fds, 'o-', color=C['clay'], ms=4, label='ảnh sinh'); axes[0].axhline(fd_real, color=C['green'], ls='--', label='ảnh train thật (mức sàn)')
axes[0].set_xticks(range(EPOCHS_IMG + 1)); axes[0].set(xlabel='epoch', ylabel='FD so với ảnh test', title='Khoảng cách Fréchet nhỏ theo epoch', yscale='log'); axes[0].legend()
axes[1].bar(range(10), cls_share * 100, color=C['blue']); axes[1].axhline(10, color=C['gray'], ls=':'); bar_labels(axes[1], '{:.0f}')
axes[1].set(xlabel='chữ số mà CNN nhận ra', ylabel='% trong 10 000 ảnh sinh', title=f'Phân bố chữ số sau epoch {EPOCHS_IMG}', xticks=range(10))
plt.tight_layout(); plt.show()
check('FD sau train thấp hơn FD lúc chưa train', fds[-1] < fds[0])
note(f'FD giảm từ {fds[0]:.0f} (chưa train) xuống <b>{fds[-1]:.1f}</b> sau {EPOCHS_IMG} epoch ({t_mnist:.0f} giây), nhưng vẫn cao gấp {fds[-1] / fd_real:.0f} lần mức sàn {fd_real:.2f} của ảnh train thật. '
     f'Chữ số thưa nhất chiếm {cls_share.min():.1%} và dày nhất {cls_share.max():.1%} (mức đều là 10%) '
     + ('nên GAN không bỏ hẳn chữ số nào, dù tỉ lệ lệch nhau.' if cls_share.min() >= .02 else f'nên chữ số {int(cls_share.argmin())} gần như vắng mặt: dấu hiệu mode collapse.')
     + ' Generator cần một lần gọi cho mỗi ảnh.')

## 4. Diffusion thuận trên ảnh thật

Diffusion thêm nhiễu theo lịch $\beta_t$ tăng đều từ $10^{-4}$ đến $0.02$ trong 1000 bước (lịch của DDPM). Đặt $\alpha_t = 1 - \beta_t$, $\bar\alpha_t = \prod_{s \le t}\alpha_s$; không cần lặp 1000 lần, chỉ cần một công thức:

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \qquad \epsilon \sim \mathcal N(0, I).$$

Dưới đây là bốn ảnh CIFAR-10 ở các mức $t$, và hệ số tín hiệu, hệ số nhiễu theo $t$.

In [ ]:
import io, types
from PIL import Image
with quiet():
    cf = pd.read_parquet('https://huggingface.co/api/datasets/uoft-cs/cifar10/parquet/plain_text/test/0.parquet')     # 10 000 ảnh test
cifar = types.SimpleNamespace(data=np.stack([np.array(Image.open(io.BytesIO(b['bytes'])).convert('RGB')) for b in cf['img']]), targets=cf['label'].tolist())
cls_vi = {0: 'máy bay', 3: 'mèo', 7: 'ngựa', 8: 'tàu'}
pick = [int(np.flatnonzero(np.array(cifar.targets) == c)[0]) for c in cls_vi]
x0 = torch.tensor(cifar.data[pick]).permute(0, 3, 1, 2).float() / 127.5 - 1            # [-1, 1]
TT = 1000
betas = torch.linspace(1e-4, 0.02, TT); alphas = 1 - betas; abar = torch.cumprod(alphas, 0)
show_t = [0, 50, 100, 200, 400, 600, 800, 999]
torch.manual_seed(0); eps = torch.randn_like(x0)
fig, axes = plt.subplots(len(x0), len(show_t), figsize=(1.45 * len(show_t), 1.5 * len(x0) + .5))
for i in range(len(x0)):
    for j, t in enumerate(show_t):
        xt = abar[t].sqrt() * x0[i] + (1 - abar[t]).sqrt() * eps[i]
        axes[i, j].imshow(((xt + 1) / 2).clamp(0, 1).permute(1, 2, 0)); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if i == 0: axes[i, j].set_title(f't = {t}', fontsize=10)
    axes[i, 0].set_ylabel(list(cls_vi.values())[i], fontsize=10)
plt.suptitle('Cùng một ε, mức nhiễu tăng dần từ trái sang phải', y=1.0); plt.tight_layout(); plt.show()
t_half = int((abar < .5).nonzero()[0])
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(abar.sqrt(), color=C['blue'], label='hệ số tín hiệu √ᾱₜ'); ax.plot((1 - abar).sqrt(), color=C['clay'], label='hệ số nhiễu √(1−ᾱₜ)'); ax.plot(abar, color=C['gray'], ls='--', label='ᾱₜ')
ax.axvline(t_half, color=C['sand'], ls=':'); ax.text(t_half + 10, .06, f'ᾱₜ = 0.5 tại t = {t_half}', color=C['sand'], fontsize=9)
ax.set(xlabel='t', ylabel='hệ số', title='Tín hiệu giảm, nhiễu tăng theo t'); ax.legend(loc='center right')
plt.tight_layout(); plt.show()

Kiểm công thức đóng bằng số: chạy **từng bước** $x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{\beta_t}\,\epsilon_t$ cho 2000 bản sao của cùng một ảnh, rồi so trung bình và phương sai của $x_t$ với $\sqrt{\bar\alpha_t}\,x_0$ và $1 - \bar\alpha_t$.

In [ ]:
torch.manual_seed(1)
img = x0[1]; xs = img[None].repeat(2000, 1, 1, 1); res = []
for t in range(300):
    xs = alphas[t].sqrt() * xs + betas[t].sqrt() * torch.randn_like(xs)
    if t + 1 in (100, 300):
        tt = t                                                                      # chỉ số của abar tương ứng với x_{t+1}
        res.append((t + 1, abar[tt].item(), (xs.mean(0) - abar[tt].sqrt() * img).abs().max().item(), xs.var(0).mean().item(), 1 - abar[tt].item()))
chk = pd.DataFrame(res, columns=['sau bước', 'ᾱ', 'sai số lớn nhất của trung bình', 'phương sai đo được', '1 − ᾱ'])
table(chk, 'Chạy từng bước so với công thức đóng', fmt={'ᾱ': '{:.4f}', 'sai số lớn nhất của trung bình': '{:.3f}', 'phương sai đo được': '{:.4f}', '1 − ᾱ': '{:.4f}'})
e_ = torch.randn_like(x0); t_ = 400; xt_ = abar[t_].sqrt() * x0 + (1 - abar[t_]).sqrt() * e_
rec = ((xt_ - (1 - abar[t_]).sqrt() * e_) / abar[t_].sqrt() - x0).abs().max().item()
check('phương sai đo được khớp 1 − ᾱ (sai lệch dưới 5%)', bool(((chk['phương sai đo được'] / chk['1 − ᾱ'] - 1).abs() < .05).all()))
check('trung bình của x_t khớp √ᾱ·x₀ (sai số lớn nhất dưới 0.1)', bool((chk['sai số lớn nhất của trung bình'] < .1).all()))
check(f'biết đúng ε thì khôi phục x₀ từ x_t với sai số {rec:.1e}', rec < 1e-4)
note(f'Ở t = {t_half}, ᾱ vượt qua 0.5: tín hiệu và nhiễu có phương sai ngang nhau. Đến t = 999, ᾱ = {abar[-1]:.1e}, tín hiệu còn {abar[-1].sqrt():.3f} lần: ảnh chỉ còn nhiễu chuẩn, nên sampler được phép bắt đầu từ N(0, I). '
     'Dòng khôi phục x₀ dùng ε thật nên chỉ là kiểm đại số; lúc sinh, ε phải do model đoán.')

## 5. Noise predictor 2D

Train đúng quy trình của bài, trên tám cụm: chọn $x_0$, chọn $t$ ngẫu nhiên trong 100 mức, tạo $x_t$ bằng công thức đóng, MLP nhận $(x_t, t)$ và đoán $\epsilon$, loss là MSE. Lịch dùng $\beta$ tăng từ 0.002 đến 0.1 (ngắn hơn lịch 1000 bước ở trên). Mạng còn nhận thêm **nhãn** là chỉ số cụm; 25% số lần train nhãn bị thay bằng nhãn rỗng, để mục 6 dùng được guidance. Ở mục này sinh với nhãn rỗng, tức không điều kiện.

Sau khi train, lấy mẫu bằng sampler DDPM: bắt đầu từ $x_T \sim \mathcal N(0, I)$, mỗi bước

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Bigl(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat\epsilon\Bigr) + \sqrt{\beta_t}\,z,$$

với $z$ là nhiễu mới (bỏ ở bước cuối).

In [ ]:
T2 = 100
b2 = torch.linspace(2e-3, 0.1, T2); a2 = 1 - b2; ab2 = torch.cumprod(a2, 0)
FREQ = torch.tensor([1, 2, 4, 8, 16, 32, 64, 128]) * math.pi

class Eps(nn.Module):
    def __init__(s):
        super().__init__()
        s.emb = nn.Embedding(K + 1, 16)                                              # K nhãn cụm + một nhãn rỗng
        s.net = nn.Sequential(nn.Linear(2 + 16 + 16, 256), nn.SiLU(), nn.Linear(256, 256), nn.SiLU(), nn.Linear(256, 256), nn.SiLU(), nn.Linear(256, 2))
    def forward(s, x, t, c):
        f = (t.float() / T2)[:, None] * FREQ
        return s.net(torch.cat([x, torch.sin(f), torch.cos(f), s.emb(c)], 1))

torch.manual_seed(0)
net2 = Eps(); opt = torch.optim.Adam(net2.parameters(), 1e-3); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS_2D)
def noisy_batch(n):
    x0_, k = sample_real(n); t = torch.randint(0, T2, (n,)); e = torch.randn(n, 2); ab = ab2[t][:, None]
    return ab.sqrt() * x0_ + (1 - ab).sqrt() * e, t, e, k
t0 = time.time(); lh = []
for s in range(STEPS_2D):
    xt, t, e, k = noisy_batch(512)
    c = torch.where(torch.rand(512) < .25, torch.full_like(k, K), k)
    loss = F.mse_loss(net2(xt, t, c), e); opt.zero_grad(); loss.backward(); opt.step(); sch.step(); lh.append(loss.item())
t_diff2d = time.time() - t0
net2.eval()

@torch.no_grad()
def sample2d(n, label=None, w=None, snaps=()):
    # label=None: nhãn rỗng. w: hệ số guidance, ε = ε_rỗng + w (ε_nhãn − ε_rỗng); w=1 là có điều kiện thường
    x = torch.randn(n, 2); out = {T2: x.clone()}
    null = torch.full((n,), K)
    for t in reversed(range(T2)):
        tt = torch.full((n,), t)
        if label is None: e = net2(x, tt, null)
        else:
            eu, ec = net2(x, tt, null), net2(x, tt, label); e = eu + w * (ec - eu)
        x = (x - b2[t] / (1 - ab2[t]).sqrt() * e) / a2[t].sqrt() + (b2[t].sqrt() * torch.randn(n, 2) if t > 0 else 0)
        if t in snaps: out[t] = x.clone()
    out[-1] = x
    return out

SN = [T2, 70, 40, 20, 8, 0]
out = sample2d(N, snaps=SN); xd = out[-1]
ms_d = mode_stats(xd); sw_d = sw(REF, xd)
fig, axes = plt.subplots(1, len(SN), figsize=(2.6 * len(SN), 3.1))
for ax, t in zip(axes, SN):
    ring(ax, f'mức nhiễu {t}' if t else 'xong (t = 0)', out[t], C['blue'])
    if ax is not axes[0]: ax.set(ylabel='', yticklabels=[])
plt.tight_layout(); plt.show()

mse_t = []
with torch.no_grad():
    for tt in range(T2):
        x0_, k = sample_real(4096); e = torch.randn(4096, 2); xt = ab2[tt].sqrt() * x0_ + (1 - ab2[tt]).sqrt() * e
        mse_t.append(F.mse_loss(net2(xt, torch.full((4096,), tt), torch.full_like(k, K)), e).item())
mse_t = np.array(mse_t)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(sm(lh, 50), color=C['blue']); axes[0].set(xlabel='bước train', ylabel='MSE (trung bình 50 bước)', title='Loss đoán nhiễu khi train', yscale='log')
axes[1].plot(mse_t, color=C['clay'], label='noise predictor'); axes[1].axhline(1, color=C['gray'], ls='--', label='luôn đoán 0 (MSE = 1)')
axes[1].set(xlabel='mức nhiễu t', ylabel='MSE đoán ε', title='Sai số đoán nhiễu theo từng mức, nhãn rỗng'); axes[1].legend(loc='lower right')
plt.tight_layout(); plt.show()
check('noise predictor tốt hơn đoán 0 ở mọi mức t', bool((mse_t < 1).all()))
check('sampler bắt đầu từ N(0, I): trung bình gần 0, độ lệch chuẩn gần 1', abs(out[T2].mean().item()) < .1 and abs(out[T2].std().item() - 1) < .1)
hard = int(mse_t.argmax())
note(f'MSE trung bình {mse_t.mean():.2f} (đoán 0 cho 1.00); khó nhất ở t = {hard} (MSE {mse_t.max():.2f}), dễ nhất ở t = {int(mse_t.argmin())} (MSE {mse_t.min():.3f}). '
     f'Sampler dùng {T2} lần gọi mạng cho mỗi mẫu và đưa 2000 điểm nhiễu về <b>{ms_d["modes"]}/8</b> cụm, {ms_d["good"]:.0%} mẫu tốt, SW = {sw_d:.3f} (sàn {SW0:.3f}); train mất {t_diff2d:.0f} giây. '
     + (f'Mẫu còn nằm nhiều giữa các cụm (chỉ {ms_d["good"]:.0%} là mẫu tốt); nhãn rỗng chỉ được train 25% số lần, còn mục 6 cho thấy khi có nhãn thì chính mạng này sinh rất chính xác. ' if ms_d['good'] < .6 else 'Phần lớn mẫu nằm gọn trong các cụm. ')
     + f'GAN ở mục 2 phủ {ms_g["modes"]}/8 cụm; so sánh đầy đủ ở mục 9.')

## 6. Sinh có điều kiện và guidance

Cùng mạng ở mục 5, đưa nhãn cụm vào. **Classifier-free guidance** gọi mạng hai lần mỗi bước, có nhãn $c$ và nhãn rỗng $\varnothing$:

$$\hat\epsilon = \epsilon_\theta(x_t,t,\varnothing) + w\,\bigl[\epsilon_\theta(x_t,t,c) - \epsilon_\theta(x_t,t,\varnothing)\bigr].$$

$w = 0$ là không điều kiện, $w = 1$ là có điều kiện thường, $w$ lớn hơn thì đẩy mạnh hơn về nhãn. Mỗi mẫu được gán một nhãn ngẫu nhiên; đo hai thứ: tỉ lệ mẫu có **tâm gần nhất là tâm của nhãn** và **khoảng cách trung bình bình phương gốc (RMS)** từ mẫu đến tâm cụm đúng. Dữ liệu thật có RMS bằng $\sigma\sqrt 2 = 0.141$.

In [ ]:
torch.manual_seed(5)
lab = torch.randint(0, K, (N,)); WS = [0, 1, 3, 6]
rows, outs = [], {}
for w in WS:
    x = sample2d(N, lab, w)[-1]; outs[w] = x
    d = (x - CEN[lab]).norm(dim=1)
    rows.append((w, (torch.cdist(x, CEN).argmin(1) == lab).float().mean().item(), d.pow(2).mean().sqrt().item(), mode_stats(x)['modes']))
cf = pd.DataFrame(rows, columns=['w', 'gần đúng cụm của nhãn', 'RMS tới tâm đúng', 'mode phủ'])
table(cf, 'Guidance trên tám cụm', fmt={'w': '{:.0f}', 'gần đúng cụm của nhãn': '{:.0%}', 'RMS tới tâm đúng': '{:.3f}'}, best='gần đúng cụm của nhãn')
cm = plt.get_cmap('tab10')
fig, axes = plt.subplots(1, len(WS), figsize=(2.9 * len(WS), 3.3))
for ax, w, (_, r) in zip(axes, WS, cf.iterrows()):
    ring(ax, f'w = {w}: {r.iloc[1]:.0%} gần đúng cụm', None)
    ax.scatter(*outs[w].T, s=3, alpha=.6, c=[cm(int(l)) for l in lab], lw=0)
    if w != WS[0]: ax.set(ylabel='', yticklabels=[])
plt.tight_layout(); plt.show()
check('w = 0 bỏ qua nhãn: tỉ lệ gần đúng cụm xấp xỉ 1/8', abs(cf.iloc[0, 1] - 1 / K) < .06)
check('w = 1 đưa phần lớn mẫu về đúng cụm của nhãn', cf.iloc[1, 1] > .6)
mono = bool(np.all(np.diff(cf['RMS tới tâm đúng'][1:]) < 0))
note(f'Không điều kiện (w = 0): {cf.iloc[0, 1]:.0%} mẫu gần cụm của nhãn nhất' + (', đúng mức ngẫu nhiên 1/8' if abs(cf.iloc[0, 1] - 1 / K) < .06 else ', lệch khỏi mức ngẫu nhiên 1/8 vì mẫu không đều giữa các cụm') + f'. Có điều kiện (w = 1): {cf.iloc[1, 1]:.0%}; w = 6: {cf.iloc[3, 1]:.0%}. '
     f'RMS từ {cf.iloc[1, 2]:.3f} (w = 1) xuống {cf.iloc[3, 2]:.3f} (w = 6), dữ liệu thật là 0.141: '
     + ('w lớn làm mỗi cụm co hẹp hơn dữ liệu thật, tức mẫu giống nhau hơn. Đó là giá phải trả: bám nhãn hơn, đa dạng ít hơn.' if cf.iloc[3, 2] < .141 and mono else
        'trong lần chạy này w lớn không co cụm rõ rệt, nên cái giá về đa dạng nhỏ hơn bài nói; thử đẩy w cao hơn để thấy.'))

## 7. DDPM có sẵn khử nhiễu

Phần còn lại dùng model thật: `google/ddpm-cifar10-32`, một U-Net khoảng 36 triệu tham số, đã train 800 nghìn bước trên CIFAR-10 với đúng lịch nhiễu ở mục 4. Ta không dùng pipeline đóng gói mà tự chạy vòng lặp: U-Net đoán $\epsilon$, scheduler của `diffusers` biến dự đoán đó thành $x_{t-1}$.

Kiểm trước: đưa vào U-Net ảnh CIFAR-10 thật đã thêm nhiễu bằng công thức đóng, so $\epsilon$ model đoán với $\epsilon$ thật.

In [ ]:
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
with quiet():
    unet = UNet2DModel.from_pretrained('google/ddpm-cifar10-32').to(device).eval()
    sched_ddpm = DDPMScheduler.from_pretrained('google/ddpm-cifar10-32')
nparams = sum(p.numel() for p in unet.parameters()) / 1e6
check(f'U-Net có {nparams:.1f} triệu tham số, nhận và trả ảnh 3 × 32 × 32', 30 < nparams < 40 and unet.config.sample_size == 32 and unet.config.in_channels == 3)
check('scheduler có 1000 mức nhiễu, β từ 1e-4 đến 0.02', sched_ddpm.config.num_train_timesteps == 1000 and abs(sched_ddpm.betas[-1].item() - .02) < 1e-6)

xr = torch.tensor(cifar.data[100:164]).permute(0, 3, 1, 2).float().div(127.5).sub(1).to(device)
rows = []
with torch.no_grad():
    torch.manual_seed(2)
    for t in [10, 100, 300, 600, 900]:
        e = torch.randn_like(xr); ab = sched_ddpm.alphas_cumprod[t].item()
        p = unet(ab ** .5 * xr + (1 - ab) ** .5 * e, t).sample
        rows.append((t, ab, F.mse_loss(p, e).item()))
pe = pd.DataFrame(rows, columns=['mức nhiễu t', 'ᾱₜ', 'MSE đoán ε'])
table(pe, 'U-Net đoán nhiễu trên 64 ảnh CIFAR-10 thật (đoán 0 cho MSE ≈ 1)', fmt={'ᾱₜ': '{:.4f}', 'MSE đoán ε': '{:.4f}'})
check('MSE đoán ε thấp hơn nhiều so với đoán 0 ở mọi mức đã thử', bool((pe['MSE đoán ε'] < .5).all()))
note(f'MSE từ {pe["MSE đoán ε"].min():.4f} (t = {int(pe.loc[pe["MSE đoán ε"].idxmin(), "mức nhiễu t"])}) đến {pe["MSE đoán ε"].max():.4f} (t = {int(pe.loc[pe["MSE đoán ε"].idxmax(), "mức nhiễu t"])}): '
     + ('đoán nhiễu khó nhất ở mức nhiễu thấp, đúng như bài nói: khi tín hiệu còn nhiều thì khó tách nhiễu ra khỏi chi tiết ảnh, còn ở mức cao ε gần như đọc thẳng từ x_t.' if pe.loc[pe['MSE đoán ε'].idxmax(), 'mức nhiễu t'] <= 300 else
        'khó nhất ở mức giữa hoặc cao, khác với mô hình 1 chiều ở bài.'))

Giờ chạy sampler DDPM đầy đủ từ nhiễu chuẩn. Mỗi bước ghi lại $x_t$ và **ước lượng $\hat x_0$** mà model suy ra từ $\hat\epsilon$ tại bước đó, $\hat x_0 = (x_t - \sqrt{1-\bar\alpha_t}\,\hat\epsilon)/\sqrt{\bar\alpha_t}$. Hàng trên là thứ sampler đang cầm, hàng dưới là "ảnh sạch mà model đoán lúc này".

In [ ]:
def sync():
    if device == 'cuda': torch.cuda.synchronize()

@torch.no_grad()
def run(sched, steps, x_init, seed=0, rec_frac=()):
    sched.set_timesteps(steps); g = torch.Generator(device=device).manual_seed(seed)
    x = x_init.clone(); n = len(sched.timesteps); at = {int(round(f * (n - 1))) for f in rec_frac}; rec = {}
    for i, t in enumerate(sched.timesteps):
        e = unet(x, t).sample; o = sched.step(e, t, x, generator=g)
        if i in at: rec[int(t)] = (x.clone(), o.pred_original_sample.clone())
        x = o.prev_sample
    return x.clamp(-1, 1), rec

def im(x): return ((x.cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
torch.manual_seed(0)
xT = torch.randn(N_IMG, 3, 32, 32, device=device)
sync(); t0 = time.time()
x_ddpm, rec = run(sched_ddpm, T_DDPM, xT, rec_frac=(0, .2, .4, .6, .8, .9, .95, 1)); sync(); t_ddpm = time.time() - t0
ts = sorted(rec, reverse=True); show_n = 3
fig, axes = plt.subplots(2 * show_n, len(ts), figsize=(1.35 * len(ts), 1.4 * 2 * show_n + .5))
for j, t in enumerate(ts):
    for i in range(show_n):
        axes[2 * i, j].imshow(im(rec[t][0][i])); axes[2 * i + 1, j].imshow(im(rec[t][1][i]))
    axes[0, j].set_title(f't = {t}', fontsize=10)
for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
for i in range(show_n): axes[2 * i, 0].set_ylabel('x_t', fontsize=10); axes[2 * i + 1, 0].set_ylabel('x̂₀', fontsize=10)
plt.suptitle('DDPM từ nhiễu chuẩn: x_t (hàng trên) và ảnh sạch model đoán (hàng dưới)', y=1.0); plt.tight_layout(); plt.show()
v0 = rec[ts[0]][0].float().std().item(); vl = x_ddpm.std().item()
check('ảnh cuối nằm trong [-1, 1] và không còn là nhiễu chuẩn (độ lệch chuẩn thay đổi)', x_ddpm.abs().max().item() <= 1 and abs(vl - v0) > .05)
check(f'x_T đầu vào có độ lệch chuẩn {v0:.2f}, gần 1', abs(v0 - 1) < .1)
note(f'{len(unet.config.block_out_channels)} cấp độ phân giải trong U-Net, {T_DDPM} lần gọi cho {N_IMG} ảnh, mất {t_ddpm:.1f} giây. '
     'Ở các mức t lớn, x̂₀ chỉ là mảng màu mờ (hoặc còn lẫn nhiễu ở mức cao nhất): model chưa biết ảnh sẽ là gì nên đoán một thứ trung bình. Càng gần t = 0, x̂₀ và x_t càng giống nhau và hiện chi tiết.')

## 8. Số bước lấy mẫu

DDPM cần 1000 lần gọi U-Net cho mỗi ảnh vì phải đi theo đúng chuỗi Markov đã train. **DDIM** dùng cùng U-Net nhưng đi theo một quỹ đạo tất định, bỏ qua nhiều mức: 50, 20 hoặc 10 bước thay vì 1000. Cùng một $x_T$ (hàng đầu của mỗi cột là cùng một nhiễu khởi đầu), cùng model, chỉ đổi sampler và số bước.

In [ ]:
res = {'DDPM': (T_DDPM, x_ddpm, t_ddpm)}
for s in (50, 20, 10):
    sched = DDIMScheduler.from_config(sched_ddpm.config)
    sync(); t0 = time.time(); xs_, _ = run(sched, s, xT); sync(); res[f'DDIM {s}'] = (s, xs_, time.time() - t0)
ref = res['DDIM 50'][1]
tb = pd.DataFrame([(k, v[0], v[2], v[2] / N_IMG * 1000, (v[1] - ref).pow(2).mean().sqrt().item() if k != 'DDIM 50' else np.nan) for k, v in res.items()],
                  columns=['sampler', 'lần gọi U-Net', 'tổng thời gian (giây)', 'ms mỗi ảnh', 'RMS lệch so với DDIM 50'])
table(tb, f'Bốn cách lấy mẫu cho {N_IMG} ảnh từ cùng nhiễu khởi đầu', fmt={'tổng thời gian (giây)': '{:.2f}', 'ms mỗi ảnh': '{:.0f}', 'RMS lệch so với DDIM 50': '{:.3f}'})
fig, axes = plt.subplots(len(res), N_IMG, figsize=(1.45 * N_IMG, 1.5 * len(res)))
for r, (k, v) in enumerate(res.items()):
    for j in range(N_IMG): axes[r, j].imshow(im(v[1][j])); axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
    axes[r, 0].set_ylabel(f'{k}\n{v[0]} bước', fontsize=10)
plt.suptitle('Cùng nhiễu khởi đầu, bốn sampler', y=1.0); plt.tight_layout(); plt.show()
check('mọi ảnh sinh nằm trong [-1, 1]', all(v[1].abs().max().item() <= 1 for v in res.values()))
check('số lần gọi U-Net đúng bằng số bước', [v[0] for v in res.values()] == [T_DDPM, 50, 20, 10])
d = tb['RMS lệch so với DDIM 50'].dropna().tolist()
note(f'DDIM 10 bước nhanh hơn DDPM khoảng <b>{tb.iloc[0, 2] / tb.iloc[3, 2]:.0f} lần</b> (số lần gọi U-Net giảm {tb.iloc[0, 1] / tb.iloc[3, 1]:.0f} lần). '
     + (f'So với DDIM 50, ảnh của DDIM 20 lệch {d[1]:.3f} và DDIM 10 lệch {d[2]:.3f}: bớt bước thì quỹ đạo đi xa dần, đúng với việc bước thô hơn thì sai số tích lũy nhiều hơn. ' if d[2] > d[1] else
        f'Ảnh của DDIM 20 và DDIM 10 lệch DDIM 50 lần lượt {d[1]:.3f} và {d[2]:.3f}, không tăng dần theo ít bước. ')
     + f'DDPM lệch {d[0]:.3f} vì mỗi bước thêm nhiễu mới, nên không phải một bản xấp xỉ của quỹ đạo tất định. '
     f'Đây là so khoảng cách pixel với một ảnh tham chiếu, chưa phải thước đo chất lượng; nhìn lưới ảnh để quyết định ảnh còn dùng được không. Generator MNIST ở mục 3: một lần gọi, {ms_img_gan:.3f} ms mỗi ảnh, khác model và khác cỡ ảnh nên chỉ so được số lần gọi.')

## 9. Đánh giá model sinh

Ba phép đo, ba bài học:

1. **Đặt cạnh nhau** GAN và diffusion 2D trên cùng thước đo.
2. **Chọn lọc** làm ảnh trông đẹp hơn thực tế: lấy 4 mẫu "đẹp nhất" trong 200 so với 4 mẫu đầu tiên.
3. **FD phụ thuộc số mẫu**: cùng một tập ảnh thật, tính FD với ít mẫu cho số lớn hơn, nên hai con số tính trên số mẫu khác nhau không so được.

In [ ]:
cmp_ = pd.DataFrame([('thật, tập khác', 8, 1.0, SW0, 1), ('GAN (mục 2)', ms_g['modes'], ms_g['good'], sw_g, 1), ('diffusion (mục 5)', ms_d['modes'], ms_d['good'], sw_d, T2)],
                    columns=['model', 'mode phủ (/8)', 'mẫu tốt', 'SW so với tập thật', 'lần gọi mạng mỗi mẫu'])
cmp_.loc[0, 'mẫu tốt'] = mode_stats(REF2)['good']
table(cmp_, 'GAN và diffusion trên tám cụm', fmt={'mẫu tốt': '{:.0%}', 'SW so với tập thật': '{:.3f}'}, best='SW so với tập thật', lower_is_better=True)

# ảnh chọn lọc: lấy snapshot GAN nhiều lỗi nhất sau bước đầu
worst = min((s for s in SNAP if s >= .5 * STEPS_2D), key=lambda s: mode_stats(snaps[s])['modes'])
pool = snaps[worst][:200]; dist = torch.cdist(pool, CEN).min(1).values
best4, first4 = pool[dist.argsort()[:4]], pool[:4]
dfull = mode_stats(snaps[worst])
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
ring(axes[0], f'4 mẫu đẹp nhất trong 200\n(GAN bước {worst})', best4, C['clay'], s=40)
ring(axes[1], '4 mẫu đầu tiên (seed cố định)', first4, C['blue'], s=40)
ring(axes[2], f'2000 mẫu cùng GAN: {dfull["modes"]}/8 cụm', snaps[worst], C['gray'])
plt.tight_layout(); plt.show()
db, df4 = torch.cdist(best4, CEN).min(1).values.mean().item(), torch.cdist(first4, CEN).min(1).values.mean().item()
modes4 = len(set(torch.cdist(best4, CEN).argmin(1).tolist()))

sizes = [100, 500, 2000, 10000]
rng = np.random.default_rng(0)
fd_n = [np.mean([fd(F_tr[rng.choice(10000, n, replace=False)], F_te) for _ in range(3)]) for n in sizes]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(sizes, fd_n, 'o-', color=C['clay']); ax.set_xscale('log'); ax.set_yscale('log')
for n, v in zip(sizes, fd_n): ax.annotate(f'{v:.2f}', (n, v), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax.margins(y=.2); ax.set(xlabel='số ảnh thật dùng để tính FD', ylabel='FD so với 10 000 ảnh test', title='Cùng một tập ảnh thật, số mẫu khác nhau cho FD khác nhau')
plt.tight_layout(); plt.show()
check('FD của cùng tập ảnh thật giảm khi dùng nhiều mẫu hơn', fd_n[0] > fd_n[-1])
better = 'diffusion' if sw_d < sw_g else 'GAN'
note(f'Trên tám cụm, {better} có SW thấp hơn ({min(sw_g, sw_d):.3f} so với {max(sw_g, sw_d):.3f}, sàn {SW0:.3f}); GAN phủ {ms_g["modes"]}/8 cụm, diffusion phủ {ms_d["modes"]}/8, nhưng diffusion tốn {T2} lần gọi mạng cho mỗi mẫu còn GAN tốn 1. '
     f'Ảnh chọn lọc (GAN bước {worst}): 4 mẫu đẹp nhất cách tâm trung bình {db:.3f} so với {df4:.3f} của 4 mẫu đầu, và chạm {modes4} cụm; '
     + (f'cùng GAN ấy chỉ phủ {dfull["modes"]}/8 cụm khi đếm 2000 mẫu, điều mà 4 ảnh đẹp không cho thấy. ' if dfull['modes'] < 8 else 'lần này GAN phủ đủ 8 cụm nên chọn lọc không che lỗi nào, nhưng vẫn làm số đẹp hơn thực tế. ')
     + f'FD của ảnh thật tụt từ {fd_n[0]:.1f} (100 ảnh) xuống {fd_n[-1]:.2f} (10 000 ảnh): số mẫu ít làm FD cao hơn mà model chẳng đổi. '
     'Giới hạn: SW và mode phủ chỉ đúng cho dữ liệu 2D này, FD ở đây dựa trên CNN nhỏ của MNIST, và FD chuẩn (Inception) cần hàng nghìn ảnh; vài chục ảnh đẹp không chứng minh được model sinh tốt.')

## 10. Tóm tắt

In [ ]:
summ = pd.DataFrame([
    ('1', 'sàn SW giữa hai tập thật 2000 mẫu', f'{SW0:.3f}'),
    ('2', f'GAN 2D sau {STEPS_2D} bước: mode phủ / SW', f'{ms_g["modes"]}/8 · {sw_g:.3f}'),
    ('3', f'GAN MNIST: FD epoch 0 → epoch {EPOCHS_IMG} (sàn ảnh thật)', f'{fds[0]:.0f} → {fds[-1]:.1f} ({fd_real:.2f})'),
    ('4', f'ᾱ ở t = 999 (lịch 1000 bước) · t để ᾱ < 0.5', f'{abar[-1]:.1e} · {t_half}'),
    ('5', f'diffusion 2D ({T2} bước): mode phủ / SW', f'{ms_d["modes"]}/8 · {sw_d:.3f}'),
    ('6', 'guidance: rơi đúng cụm khi w = 0 → 1 → 6', f'{cf.iloc[0, 1]:.0%} → {cf.iloc[1, 1]:.0%} → {cf.iloc[3, 1]:.0%}'),
    ('7', 'DDPM CIFAR-10 có sẵn: MSE đoán ε thấp nhất / cao nhất', f'{pe["MSE đoán ε"].min():.4f} / {pe["MSE đoán ε"].max():.3f}'),
    ('8', f'thời gian {N_IMG} ảnh: DDPM {T_DDPM} bước · DDIM 10 bước', f'{t_ddpm:.1f} s · {res["DDIM 10"][2]:.2f} s'),
    ('9', 'FD ảnh thật với 100 → 10 000 mẫu', f'{fd_n[0]:.1f} → {fd_n[-1]:.2f}'),
], columns=['mục', 'số đo', 'giá trị'])
table(summ, 'Các con số chính')
check('mọi mục đều có số', len(summ) == 9)

**Thử thêm khi dạy:** đổi `STEPS_2D` xuống 1500 để thấy cả hai model thiếu cụm; đổi `EPOCHS_IMG` lên 30 xem FD của GAN MNIST còn giảm không; đặt `T_DDPM = 250` rồi so lại bốn sampler; hoặc đổi `WS` ở mục 6 thành `[0, 1, 6, 15]` để xem $w$ rất lớn làm cụm co lại thế nào.